## Data Visualization

prepare data for visualization  
anlyiz number of rides by  
  season  
  month  
compare with weather conditions  
Visualize Top Start Stations  
Visualize Top End Stations  
Homework Part 2"

In [21]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile
import statsmodels.api as sm

In [3]:
citibike_df = pd.read_csv("../data/JC/JC2025_Enriched.csv")

citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,date,month,month_name,day_of_week,hour,season,ride_duration_minutes
0,04CF7A399050E404,classic_bike,2025-02-22 17:40:16.500,2025-02-22 17:47:22.479,Jersey & 3rd,JC074,Van Vorst Park,JC035,40.723332,-74.045953,40.718489,-74.047727,casual,2025-02-22,2025-02,February,Saturday,17,Winter,7.09965
1,124AC7493E82D845,classic_bike,2025-02-21 12:28:13.319,2025-02-21 12:35:44.762,Jersey & 3rd,JC074,Columbus Drive,JC014,40.723332,-74.045953,40.718355,-74.038914,member,2025-02-21,2025-02,February,Friday,12,Winter,7.52405
2,1A3BCA838E968327,classic_bike,2025-02-01 14:17:43.272,2025-02-01 14:59:09.894,Jersey & 3rd,JC074,Grove St PATH,JC115,40.723332,-74.045953,40.719410,-74.043090,casual,2025-02-01,2025-02,February,Saturday,14,Winter,41.44370
3,5994017EE989D6EE,electric_bike,2025-02-22 11:36:29.292,2025-02-22 11:49:51.531,Jersey & 3rd,JC074,Jersey & 3rd,JC074,40.723332,-74.045953,40.723332,-74.045953,casual,2025-02-22,2025-02,February,Saturday,11,Winter,13.37065
4,F81BCB97915C6BE6,electric_bike,2025-02-28 22:56:26.546,2025-02-28 23:07:40.391,Jersey & 3rd,JC074,Monroe St & 11 St,HB508,40.723332,-74.045953,40.750109,-74.036637,casual,2025-02-28,2025-02,February,Friday,22,Winter,11.23075


In [4]:
monthly_rides = (
    citibike_df
    .groupby("month", as_index=False)
    .agg(number_of_rides=("ride_id", "count"))
)

monthly_rides

,month,number_of_rides
0,2025-01,1
1,2025-02,45250
2,2025-03,73277
3,2025-04,81533
4,2025-05,93202
5,2025-06,96736
6,2025-07,107374
7,2025-08,108001
8,2025-09,115580
9,2025-10,103586


In [7]:
fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides"
)

fig.show()

In [8]:
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,93113
1,Spring,248012
2,Summer,312111
0,Autumn,295399


In [9]:
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

#### Top 10 Start Stations

In [10]:
top_start_stations = (
    citibike_df
    .dropna(subset=["start_station_name"])
    .groupby("start_station_name", as_index=False)
    .agg(
        number_of_departures=("ride_id", "count")
    )
    .sort_values("number_of_departures", ascending=False)
    .head(10)
)

top_start_stations

,start_station_name,number_of_departures
52,Grove St PATH,42388
58,Hoboken Terminal - Hudson St & Hudson Pl,24723
93,River St & Newark St,21383
53,Hamilton Park,21139
84,Newport PATH,19554
18,Bergen Ave & Sip Ave,19116
44,Exchange Pl,19019
0,11 St & Washington St,18593
92,River St & 1 St,18031
2,14 St Ferry - 14 St & Shipyard Ln,17881


In [11]:
fig = px.bar(
    top_start_stations.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

#### Top 10 End Stations

In [12]:
top_end_stations = (
    citibike_df
    .dropna(subset=["end_station_name"])
    .groupby("end_station_name", as_index=False)
    .agg(
        number_of_arrivals=("ride_id", "count")
    )
    .sort_values("number_of_arrivals", ascending=False)
    .head(10)
)

top_end_stations

,end_station_name,number_of_arrivals
229,Grove St PATH,44918
238,Hoboken Terminal - Hudson St & Hudson Pl,25505
340,River St & Newark St,22113
230,Hamilton Park,21230
311,Newport PATH,19592
204,Exchange Pl,19119
71,Bergen Ave & Sip Ave,19079
7,11 St & Washington St,18610
312,Newport Pkwy,17846
10,14 St Ferry - 14 St & Shipyard Ln,17619


In [13]:
fig = px.bar(
    top_end_stations.sort_values("number_of_arrivals"),
    x="number_of_arrivals",
    y="end_station_name",
    orientation="h",
    title="Top 10 End Stations by Number of Arrivals",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Arrivals",
    yaxis_title="End Station"
)

fig.show()

#### Merge with Weather Data to See Patterns

In [14]:
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2025-01-31,1
1,2025-02-01,1437
2,2025-02-02,1101
3,2025-02-03,1795
4,2025-02-04,2060


In [16]:
weather_daily = pd.read_csv("../data/JC/jersey_weather_2025.csv")

weather_daily["date"] = pd.to_datetime(weather_daily["date"])

weather_daily.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,11.0,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


In [17]:
bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2025-01-31,1,6.2,0.7,3.6,10.3,10.3,0.00,13.5
1,2025-02-01,1437,6.2,-5.8,0.7,2.4,2.4,0.00,19.4
2,2025-02-02,1101,1.4,-9.6,-4.5,1.2,0.0,0.84,11.7
3,2025-02-03,1795,6.1,-3.1,1.9,0.3,0.0,0.21,12.8
4,2025-02-04,2060,6.2,-0.5,3.6,0.3,0.3,0.00,23.0


In [18]:
bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     0
temperature_2m_min     0
temperature_2m_mean    0
precipitation_sum      0
rain_sum               0
snowfall_sum           0
wind_speed_10m_max     0
dtype: int64

In [22]:
fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

In [23]:
fig = px.scatter(
    bike_weather_daily,
    x="wind_speed_10m_max",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Maximum Wind Speed"
)

fig.update_layout(
    xaxis_title="Maximum Wind Speed",
    yaxis_title="Number of Rides"
)

fig.show()

In [24]:
fig = px.scatter(
    bike_weather_daily,
    x="precipitation_sum",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Precipitation"
)

fig.update_layout(
    xaxis_title="Daily Precipitation",
    yaxis_title="Number of Rides"
)

fig.show()

In [25]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["number_of_rides"],
        mode="lines",
        name="Daily Rides",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["temperature_2m_mean"],
        mode="lines",
        name="Daily Average Temperature",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Daily Rides and Daily Average Temperature",
    xaxis=dict(
        title="Day"
    ),
    yaxis=dict(
        title="Daily Rides",
        side="left"
    ),
    yaxis2=dict(
        title="Daily Average Temperature",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()